# 🐿️ Central Park Squirrel Census (Stories) - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook ingests the **2018 Central Park Squirrel Census - Stories** into Elasticsearch: short field notes written by volunteers who watched the park's squirrels.

**What this notebook does:**
1. Fetches the stories from [NYC Open Data](https://data.cityofnewyork.us/Environment/2018-Central-Park-Squirrel-Census-Stories/gfqj-f768) (dataset `gfqj-f768`) - no API key
2. Normalises each story (parses the date, keeps the narrative note)
3. Bulk-indexes everything into `nyc_squirrel_stories`
4. **(Optional)** adds a Mistral-powered `semantic_text` field so you can search the stories by *meaning*

The narrative notes are the fun part - weird, charming, and perfect for semantic search with Mistral embeddings.

---
> **Data source:** 2018 Central Park Squirrel Census Stories, `gfqj-f768`, Socrata JSON API.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from datetime import datetime
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
Find your endpoint and API key in:
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch the squirrel stories
Public Socrata JSON API - no key needed. The full set is small (a few hundred stories), so we grab them all.

In [ ]:
DATA_URL = 'https://data.cityofnewyork.us/resource/gfqj-f768.json'
resp = requests.get(DATA_URL, params={'$limit': 5000}, timeout=60)
resp.raise_for_status()
rows = resp.json()

print(f'✅ Fetched {len(rows)} stories')
print()
print('A few notes:')
for r in rows[:3]:
    print(' •', r.get('note_squirrel_park_stories', '')[:100])

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'nyc_squirrel_stories'

mapping = {
    'mappings': {
        'properties': {
            'hectare':      {'type': 'keyword'},   # grid cell in Central Park, e.g. 01A
            'shift':        {'type': 'keyword'},   # AM / PM
            'date':         {'type': 'date'},
            'note':         {'type': 'text'},      # the story itself
            'story_topic':  {'type': 'boolean'},
        }
    }
}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')

es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest
The raw `date` is a `MMDDYYYY` string (e.g. `10142018`); we parse it to a real date.

In [ ]:
def parse_date(d):
    try:
        return datetime.strptime(str(d), '%m%d%Y').strftime('%Y-%m-%d')
    except (TypeError, ValueError):
        return None

def enrich(r):
    doc = {
        'hectare':     r.get('hectare'),
        'shift':       r.get('shift'),
        'date':        parse_date(r.get('date')),
        'note':        r.get('note_squirrel_park_stories'),
        'story_topic': r.get('story_topic_squirrel'),
    }
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich(r) for r in rows if r.get('note_squirrel_park_stories')]
actions = [{'_index': INDEX, '_source': d} for d in docs]

success, errors = helpers.bulk(es, actions, raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} stories ({len(errors)} errors)')

## 5. Verify

In [ ]:
resp = es.search(index=INDEX, size=0, body={
    'aggs': {'by_shift': {'terms': {'field': 'shift'}}}
})
print(f"Total stories: {resp['hits']['total']['value']}")
for b in resp['aggregations']['by_shift']['buckets']:
    print(f"  {b['key']}: {b['doc_count']}")

# A keyword search to prove the text field works
resp = es.search(index=INDEX, size=3, body={'query': {'match': {'note': 'peanut'}}})
print('\nStories mentioning "peanut":')
for h in resp['hits']['hits']:
    print(' •', h['_source']['note'][:120])

## 6. Going further with Mistral

The narrative notes are perfect for **Mistral embeddings**: add a `semantic_text` field so you can search the stories by *meaning* ("a squirrel acting aggressive") instead of keywords, or feed them to a Mistral chat model for summaries. See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).

---
## ✅ Index loaded
`nyc_squirrel_stories` is ready. Add Mistral semantic search over the stories with [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).